# Project Data Prep Notebook
**Data cleaning steps performed**
- Missing data handling
- Outlier handling
- Error correction
- Duplicate removal


**Feature engineering decisions**
- Single variable transforms
- New feature definitions
- Categorical encoding decisions
- Normalization/standardization decisions
- Date/time transformations 


**Feature selection decisions**
- Variable to remove and rationale


**Dataset partitioning decisions (oversampling/undersampling)**


**Final dataset summary**
- Number of rows/columns
- Summary of key variables
- Summary of transformations and cleaning steps performed


In [ ]:
#imports 
import pandas as pd
import numpy as np
import scipy.stats as stats

In [ ]:
df_raw = pd.read_csv(
    "../data/interim/healthcare_readmissions_dataset_train_post_eda.csv",
    keep_default_na=False,
    na_values=[""],
)

In [ ]:
""" 
loading EDA dataset; has age outliers removed but thats it 
"""

df_raw.head()

## **DATA CLEANING**

### Missing data handling

In [ ]:
df_raw.info() # only missings in number_of_prior_visits and medications_prescribed

In [ ]:
# check for overlaps in missings

overlap = df_raw['number_of_prior_visits'].isna() & df_raw['medications_prescribed'].isna()
both_missing = df_raw[df_raw['number_of_prior_visits'].isna() & df_raw['medications_prescribed'].isna()]

print(f"Number of rows with both 'number_of_prior_visits' and 'medications_prescribed' missing: {both_missing.shape[0]}")




Not a lot of overlap in missingness, which means removing all the rows would probably not work.

Will return to imputing later as columns will be changed potentially. 

### Outlier handling

In [ ]:
# looking into potential bmi outliers

threshold = 3.5
z_scores = np.abs(stats.zscore(df_raw['bmi']))
outliers = df_raw[z_scores > threshold]
print(f"Identified {len(outliers)} BMI outliers at threshold {threshold}:")

display(outliers)

I think these rows are safe to delete, the bmis are just too extreme to be worth considering. 

In [ ]:
df_transformed = df_raw.drop(outliers.index)

In [ ]:
df_transformed.info() # now at 7951 x 19 columns

EDA determined there were no duplicates, and also got rid of the obvious "errors" (impossible ages). 

## **FEATURE ENGINEERING**

### Single Variable Transforms

In [ ]:
# turning medications_prescribed to binary

df_transformed = df_transformed.copy() # to avoid SettingWithCopyWarning

df_transformed["medications_prescribed"] = df_transformed["medications_prescribed"].replace("", pd.NA).astype(float)
df_transformed["is_prescribed"] = df_transformed["medications_prescribed"].apply(lambda x: 1 if x > 0 else 0)
# imputing to false/0 by default for now; will be investigated later

In [ ]:
df_transformed.drop(columns=["medications_prescribed"], inplace=True)

In [ ]:
df_transformed.info()

In [ ]:
# converting to numeric
df_transformed["number_of_prior_visits"] = df_transformed["number_of_prior_visits"].replace("", pd.NA).astype(float)

In [ ]:
""" 
converting Length of Stay to a length of stay score.
Encoded the same way as used in the LACE Index (Length of stay, Acuity of admission, Comorbidity, Emergency department use),
a popular readmission risk scoring system.
"""
def add_length_of_stay_score(df: pd.DataFrame) -> pd.DataFrame:
    """Converts length_of_stay (days) to an ordinal risk score (1–7)."""
    df = df.copy()
    def _score(x):
        if x <= 1: return 1
        if x <= 2: return 2
        if x <= 3: return 3
        if x <= 6: return 4
        if x <= 14: return 5
        return 7
    df["length_of_stay_score"] = df["length_of_stay"].apply(_score)
    return df


In [ ]:
df_transformed = add_length_of_stay_score(df_transformed)

In [ ]:
df_transformed.drop(columns=["length_of_stay"], inplace=True) # length_of_stay replaced by length_of_stay_score

In [ ]:
df_transformed.info() # bmi outliers removed, missings still the same

In [ ]:
"""
binning age feature
Bins: 0–18, 19–25, 26–40, 41–65, 66–80, 81+
Rationale: age has a non-linear relationship with readmission risk.
Binning captures clinical risk tiers more effectively than raw continuous values
Raw age column dropped after binning
"""

age_bins = [0, 18, 25, 40, 65, 80, np.inf]
age_labels = ["0-18", "19-25", "26-40", "41-65", "66-80", "81+"]
df_transformed["age_group"] = pd.cut(df_transformed["age"], bins=age_bins, labels=age_labels, right=False)
df_transformed.drop(columns=["age"], inplace=True) # age replaced by age_group

In [ ]:
# dropping weight as adjusted_weight is hghly correlated and ecists for a reason (i asuszme)
df_transformed.drop(columns=["weight_kg"], inplace=True) 

In [ ]:
display(df_transformed.head())

Notes for future investigation:
- number of prior visits could potentially be imputed based on type of treatment (?) look into perhaps


### Feature Selection Decisions

- dropped weight_kg as adjusted_weight_kg is hghly correlated and I assume would not be added to the dataset for no reason.
- age and length_of_stay were transformed into scores, so the original variables were dropped.
- non-categorical data will be standardized but will be done later
- SMOTE and other oversmapling techniques will be tested in experimenting phase

## **DATASET SUMMARY**

In [ ]:
display(df_transformed.head())

In [ ]:
display(df_transformed.info())

In [ ]:
# imputing number_of_prior_visits

df_transformed["number_of_prior_visits"] = df_transformed["number_of_prior_visits"].fillna(0)

In [ ]:
df_transformed.info()

In [ ]:
df_final = df_transformed.copy()

In [ ]:
info_df = pd.DataFrame({
    "Column": df_final.columns,
    "Non-Null Count": df_final.count().values,
    "Dtype": df_final.dtypes.astype(str).values,
})
print(info_df.to_latex(index=False, caption="Dataset Info", label="tab:dataset_info"))

In [ ]:
\begin{table}
\caption{Dataset Info}
\label{tab:dataset_info}
\begin{tabular}{lrl}
\toprule
Column & Non-Null Count & Dtype \\
\midrule
patient_id & 7951 & int64 \\
gender & 7951 & str \\
ethnicity & 7951 & str \\
hospital_id & 7951 & str \\
height_m & 7951 & float64 \\
smoker & 7951 & bool \\
bmi & 7951 & float64 \\
adjusted_weight_kg & 7951 & float64 \\
has_diabetes & 7951 & int64 \\
has_hypertension & 7951 & int64 \\
exercise_frequency & 7951 & str \\
diet_type & 7951 & str \\
number_of_prior_visits & 7951 & float64 \\
type_of_treatment & 7951 & str \\
readmission_within_30_days & 7951 & int64 \\
is_prescribed & 7951 & int64 \\
length_of_stay_score & 7951 & int64 \\
age_group & 7951 & category \\
\bottomrule
\end{tabular}
\end{table}

In [ ]:
df_final.to_csv("../data/interim/healthcare_readmissions_dataset_train_preprocessed.csv", index=False)